# FIERY - planning - number of potential collisions

In [24]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.metrics import log_loss
import pickle
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import pandas as pd

In [20]:
def get_indices_split(random_state = 828, test_size = 0.8, df_file = "df_seq_nr.csv"):
    
    seq_val, seq_test = train_test_split(np.arange(1, 151), test_size=test_size, random_state=random_state)
    df_s = pd.read_csv("df_seq_nr.csv")
    
    np.random.seed(1337)
    indices_val = df_s[df_s.seq_nr.isin(seq_val)].index.tolist()
    np.random.shuffle(indices_val)
    
    indices_test = df_s[df_s.seq_nr.isin(seq_test)].index.tolist()
    np.random.shuffle(indices_test)
    
    return indices_val, indices_test


In [26]:
indices_val, indices_test = get_indices_split(test_size=0.8, random_state=1001)

### Collisions for each timestep separately

In [7]:
#collisions_cor = np.load("collisions_5meter_corridor_5timesteps_23_03_23.npy")
# "collisions_5meter_corridor_05_01_22.npy"

In [8]:
#collisions_cor = np.load("collisions_5meter_corridor_4timesteps_23_03_23.npy")


In [9]:
collisions_cor = np.load("collisions_5meter_corridor_step_23_03_23.npy")



In [3]:
# Total collisi

In [6]:
collisions_cor.sum()

8330.0

In [32]:
collisions_cor.shape

(5119, 25)

In [12]:
np.sum(collisions_cor)/(5119*25)  # 5 timesteps

0.050978706778667705

In [13]:
costmaps_cor = np.load("costmaps_5meter_corridor_test_uncal_f0_23_03_23.npy")

In [14]:
costmaps_cor_cal = np.load("costmaps_5meter_corridor_test_cal_f0_23_03_23.npy")

In [15]:
costmaps_cor_objcal = np.load("costmaps_5meter_corridor_test_objcal_f0_23_03_23.npy")

### Get undetected object ahead calibrated scores

In [16]:
costmaps_cor_uoa = np.load("results_probabilities_undetected_cal_23_03_23.npy")

In [17]:
#costmaps_cor_uoa_nomatch = np.load("results_probabilities_undetected_cal_nomatching_23_03_23.npy")
costmaps_cor_uoa_nomatch = np.load("results_probabilities_undetected_cal_nomatching_objcal_n_29_03_23.npy")

In [18]:
costmaps_cor_uoa_objcal = np.load("results_probabilities_undetected_cal_objcal_n_29_03_23.npy")

## AUCs of different results

In [27]:
aucs = []
aucs_segmcal = []
aucs_objcal = []
aucs_uoa = []
aucs_uoa_objcal = []
aucs_uoa_objcal_segmcal = []
aucs_all = []
aucs_uoa_nomatch = []

#subset = 

for col_cor, cost_cor, cost_cor_objcal, cost_cor_uoa, cost_cor_cal, cost_cor_uoa_nm, cost_cor_uoa_objcal \
in zip(collisions_cor[indices_test], costmaps_cor, costmaps_cor_objcal, costmaps_cor_uoa, costmaps_cor_cal, 
       costmaps_cor_uoa_nomatch, costmaps_cor_uoa_objcal):
    if np.mean(col_cor) != 0 and np.mean(col_cor) != 1:
        
       # if np.sum(cost_cor) == 0:
            #print("break")
            #break
        #cost_ = cost_cor_objcal #*cost_cor_cal #*cost_cor_cal
        #print(cost_cor2)
        #break
        auc = roc_auc_score(col_cor, cost_cor)
        auc2 = roc_auc_score(col_cor, cost_cor_cal)
        #auc3 = roc_auc_score(col_cor, cost_cor_objcal*0.2)
        #auc4 = roc_auc_score(col_cor, cost_cor_uoa)
        auc5 = roc_auc_score(col_cor, cost_cor_uoa_objcal + cost_cor_objcal*0.2)
        #auc6 = roc_auc_score(col_cor, cost_cor_uoa + cost_cor_objcal*0.2 + cost_cor_cal)
        #auc7 = roc_auc_score(col_cor, cost_cor_uoa + cost_cor_objcal*0.2 + cost_cor_cal + cost_cor)
        auc8 = roc_auc_score(col_cor, cost_cor_uoa_objcal)
        
        aucs.append(auc)
        aucs_segmcal.append(auc2)
        #aucs_objcal.append(auc3)
        #aucs_uoa.append(auc4)
        aucs_uoa_objcal.append(auc5)
        #aucs_uoa_objcal_segmcal.append(auc6)
        #aucs_all.append(auc7)
        aucs_uoa_nomatch.append(auc8)
#roc_auc_score(collisions_cor[indices_test].flatten(), costmaps_cor.flatten())

In [28]:
aucs_test = []

#subset = 

for col_cor, cost_cor, cost_cor_objcal, cost_cor_uoa, cost_cor_cal, cost_cor_uoa_nm in zip(collisions_cor[indices_test], 
                                                                          costmaps_cor, costmaps_cor_objcal, 
                                                                          costmaps_cor_uoa, costmaps_cor_cal,
                                                                         costmaps_cor_uoa_nomatch):
    if np.mean(col_cor) != 0 and np.mean(col_cor) != 1:
        
       # if np.sum(cost_cor) == 0:
            #print("break")
            #break
        #cost_ = cost_cor_objcal #*cost_cor_cal #*cost_cor_cal
        #print(cost_cor2)
        #break
        auc_test = roc_auc_score(col_cor, cost_cor_uoa_nm)

        aucs_test.append(auc_test)

#roc_auc_score(collisions_cor[indices_test].flatten(), costmaps_cor.flatten())

In [29]:
np.mean(aucs_test)

0.871230902717007

In [30]:
import pandas as pd

df = pd.DataFrame(columns = ["Name", "Score"])
df.loc[0] = ["Uncal", np.mean(aucs)]
df.loc[1] = ["Segm. cal.", np.mean(aucs_segmcal)]
#df.loc[2] = ["Object cal.", np.mean(aucs_objcal)]
#df.loc[3] = ["UOA cal.", np.mean(aucs_uoa)]
df.loc[4] = ["Obj. + UOA cal.", np.mean(aucs_uoa_objcal)]
#df.loc[5] = ["Obj. + UOA + segm. cal.", np.mean(aucs_uoa_objcal_segmcal)]
#df.loc[6] = ["All", np.mean(aucs_all)]
df.loc[8] = ["UOA nomatch", np.mean(aucs_uoa_nomatch)]

In [31]:
df

,Name,Score
0,Uncal,0.895200
1,Segm. cal.,0.903931
4,Obj. + UOA cal.,0.904931
8,UOA nomatch,0.882488


In [32]:
from sklearn.metrics import log_loss

In [33]:
nll_test = []

#subset = 

for col_cor, cost_cor, cost_cor_objcal, cost_cor_uoa, cost_cor_cal in zip(collisions_cor[indices_test], costmaps_cor, costmaps_cor_objcal, costmaps_cor_uoa, costmaps_cor_cal):
    #if np.mean(col_cor) != 0 and np.mean(col_cor) != 1:
        
       # if np.sum(cost_cor) == 0:
            #print("break")
            #break
        #cost_ = cost_cor_objcal #*cost_cor_cal #*cost_cor_cal
        #print(cost_cor2)
        #break
    nll = log_loss(col_cor, cost_cor_objcal*0.2+cost_cor_uoa, labels = [0, 1]) 

    nll_test.append(nll)

#roc_auc_score(collisions_cor[indices_test].flatten(), costmaps_cor.flatten())

In [34]:
nll_uncal = log_loss(collisions_cor[indices_test].flatten(), costmaps_cor.flatten())

In [35]:
nll_pwcal = log_loss(collisions_cor[indices_test].flatten(), costmaps_cor_cal.flatten())

In [108]:
nll_objcal = log_loss(collisions_cor[indices_test].flatten(), (costmaps_cor_objcal.flatten()*0.2 + costmaps_cor_uoa.flatten()))

In [109]:
nll_uoa = log_loss(collisions_cor[indices_test].flatten(), (costmaps_cor_uoa_nomatch).flatten())
nll_uoa

0.13176475632920392

In [110]:
for multip in np.arange(0.5, 1.5, 0.1):
    #print(multip)
    print("NLL at %f is %f" % (multip, log_loss(collisions_cor[indices_test].flatten(), (costmaps_cor_uoa_nomatch).flatten()*multip)))

NLL at 0.500000 is 0.138144
NLL at 0.600000 is 0.132540
NLL at 0.700000 is 0.128873
NLL at 0.800000 is 0.127059
NLL at 0.900000 is 0.127468
NLL at 1.000000 is 0.131765
NLL at 1.100000 is 0.212287
NLL at 1.200000 is 0.548076
NLL at 1.300000 is 0.652440
NLL at 1.400000 is 0.655108


### Brier score

In [111]:
brier_uncal = np.mean(np.square(collisions_cor[indices_test].flatten() - np.clip(costmaps_cor.flatten(), 0, 1)))

In [112]:
brier_pwcal = np.mean(np.square(collisions_cor[indices_test].flatten() - np.clip(costmaps_cor_cal.flatten(), 0, 1)))

In [113]:
brier_objcal = np.mean(np.square(collisions_cor[indices_test].flatten() - np.clip((costmaps_cor_objcal.flatten()*0.2 + costmaps_cor_uoa.flatten()), 0, 1)))

In [114]:
brier_uoa = np.mean(np.square(collisions_cor[indices_test].flatten() - np.clip((costmaps_cor_uoa_nomatch.flatten()), 0, 1)))

In [115]:
#nll_uoa = -1
#brier_uoa = -1

In [116]:
df = pd.DataFrame(columns = ["Uncal Segm", "Pw-cal. Segm", "UOA", "Obj. cal. + UOA"])
df.loc["NLL"] = [nll_uncal, nll_pwcal, nll_uoa, nll_objcal]
df.loc["BS"] = [brier_uncal, brier_pwcal, brier_uoa, brier_objcal]
df.loc["AUC"] = [np.mean(aucs), np.mean(aucs_segmcal), np.mean(aucs_uoa_nomatch), np.mean(aucs_uoa_objcal)]

df.T

,NLL,BS,AUC
Uncal Segm,0.628794,0.034923,0.895200
Pw-cal. Segm,0.556205,0.033925,0.903931
UOA,0.131765,0.031434,0.882488
Obj. cal. + UOA,0.480762,0.033295,0.904931


# Get number of collisions

In [47]:
len(indices_test)

4091

In [46]:
costmaps_cor.shape

(4091, 25)

In [50]:
collisions_cor[indices_test].sum()

5017.0

In [126]:
print(df.T.to_latex())

\begin{tabular}{lrrr}
\toprule
{} &       NLL &        BS &       AUC \\
\midrule
Uncal Segm      &  0.605926 &  0.034923 &  0.895200 \\
Pw-cal. Segm    &  0.536661 &  0.033925 &  0.903931 \\
UOA             &  0.128787 &  0.032141 &  0.882922 \\
Obj. cal. + UOA &  0.269738 &  0.032463 &  0.904931 \\
\bottomrule
\end{tabular}



In [60]:
cost_cor_uoa_objcal.shape

(25,)

In [69]:
costmaps_cor_uoa

array([[1.91650873e-04, 3.14325609e-05, 1.91008919e-23, ...,
        3.52080822e-03, 2.92439710e-03, 2.32861195e-03],
       [5.16814302e-03, 8.04512998e-03, 8.90812222e-08, ...,
        7.20549400e-03, 5.01708604e-03, 4.64410218e-03],
       [5.16814302e-03, 8.04512998e-03, 1.23455817e-02, ...,
        4.85187868e-03, 1.16026745e-02, 2.11343753e-02],
       ...,
       [3.05907144e-04, 8.71877160e-05, 7.42379757e-20, ...,
        2.15478985e-03, 1.95366786e-03, 1.23367521e-03],
       [1.42180566e-04, 2.67168927e-05, 1.43770516e-23, ...,
        1.07699221e-03, 1.12197002e-03, 5.83596015e-04],
       [3.07482830e-04, 6.80415183e-05, 2.09506368e-21, ...,
        4.17530891e-03, 3.19735934e-03, 2.41459190e-03]])

In [103]:
for costmap in [costmaps_cor, costmaps_cor_cal, costmaps_cor_objcal*0.2+costmaps_cor_uoa_objcal, costmaps_cor_objcal*0.9+costmaps_cor_uoa]:

    best_indices = np.argmin(costmap, axis=1)
    
    # Collision value corresponding to each selected candidate
    selected_collisions = collisions_cor[indices_test][
        np.arange(costmap.shape[0]),
        best_indices
    ]
    
    # Number of selected choices that collide
    num_collisions = np.count_nonzero(selected_collisions)
    
    print("Number of collisions:", num_collisions)

Number of collisions: 51
Number of collisions: 48
Number of collisions: 52
Number of collisions: 80


## undetected objects ahead

### Sort costmaps and collisions for uncal and cal

1) Check the count of collisions choosing best trajectory
2) Check where the predictions differ, 

In [86]:
cost_col = np.array([costmaps_cor, collisions_cor[indices_test]]).transpose(1,0,2)
cost_col_cal = np.array([costmaps_cor_cal, collisions_cor[indices_test]]).transpose(1,0,2)

In [87]:
costmaps_cor.shape

(4091, 25)

In [88]:
ind = costmaps_cor.argsort()

cost_col_sorted = []

for i in range(costmaps_cor.shape[0]):
    
    cost_col_sorted.append([costmaps_cor[i, ind[i]], collisions_cor[indices_test][i, ind[i]]])
    
cost_col_sorted = np.array(cost_col_sorted)

In [89]:
ind = costmaps_cor_cal.argsort()

cost_col_cal_sorted = []

for i in range(costmaps_cor_cal.shape[0]):
    
    cost_col_cal_sorted.append([costmaps_cor_cal[i, ind[i]], collisions_cor[indices_test][i, ind[i]]])
    
cost_col_cal_sorted = np.array(cost_col_cal_sorted)

In [90]:
cost_col_cal.shape

(4091, 2, 25)

In [91]:
indices_cc = np.where(np.logical_and(cost_col.sum(axis=(2))[:, 1] > 0, cost_col.sum(axis=(2))[:, 1] < 25))[0]

In [92]:
indices_cc[:100]

array([  2,  10,  14,  18,  19,  23,  25,  27,  28,  31,  36,  44,  45,
        48,  50,  52,  58,  59,  63,  66,  73,  75,  82,  84,  85,  86,
        87,  89,  93,  97,  99, 108, 112, 113, 117, 118, 135, 137, 150,
       162, 163, 165, 169, 170, 172, 175, 179, 180, 182, 184, 191, 196,
       202, 205, 216, 218, 220, 223, 226, 234, 238, 239, 243, 245, 246,
       248, 250, 256, 272, 277, 279, 288, 289, 298, 318, 321, 325, 328,
       332, 337, 339, 340, 345, 348, 349, 352, 353, 360, 365, 366, 369,
       372, 380, 384, 385, 393, 398, 405, 414, 416])

In [93]:
len(aucs)

920

rounding 2

In [ ]:
np.mean(np.array(aucs_cal2)[np.where((np.array(aucs2) - aucs_cal2) != 0)])

round 3

In [ ]:
np.mean((np.array(aucs2))[np.where((np.array(aucs2) - aucs_cal2) != 0)])

In [ ]:
np.mean(np.array(aucs_cal2)[np.where((np.array(aucs2) - aucs_cal2) != 0)])

raw

In [ ]:
np.mean((np.array(aucs))[np.where((np.array(aucs) - aucs_cal) != 0)])

In [ ]:
np.mean((np.array(aucs_cal))[np.where((np.array(aucs) - aucs_cal) != 0)])

## Histogram of aucs differences

Negative means calibrated better, positive uncal better

In [ ]:
np.arange(-1,1,0.05)+0.025

In [ ]:
aucs_diffs = np.array(aucs) - aucs_cal

In [ ]:
np.mean(aucs_diffs[np.where(aucs_diffs != 0)])

In [ ]:
plt.hist(aucs_diffs, bins=15)

In [ ]:
plt.hist(aucs_diffs[np.where(aucs_diffs != 0)], bins=np.arange(-1,1.1,0.1))

## Sorting costmaps and collisions

In [ ]:
np.where(np.abs(aucs_diffs) >= 0.25)

In [ ]:
idx = 188

In [ ]:
cost_col[indices_cc[idx]]

In [ ]:
np.round(cost_col_sorted[indices_cc[idx]],5)

In [ ]:
roc_auc_score(cost_col_sorted[indices_cc[idx]][1], cost_col_sorted[indices_cc[idx]][0])

In [ ]:
cost_col_cal_sorted[indices_cc[idx]]

In [ ]:
roc_auc_score(cost_col_cal_sorted[indices_cc[idx]][1], cost_col_cal_sorted[indices_cc[idx]][0])

##### Best trajectory and number of collision

In [ ]:
cost_col_sorted[:, 1, 0].sum()

In [ ]:
cost_col_cal_sorted[:, 1, 0].sum()